# Pipeline: 

## Important: use multiple core cpu (`pyarrow`) to download and read/write the dataset, since gpu enhencement doesn't seems to work on Mac silicon

|Process|Steps|Notes|
|---|---|---|
|download zip files as tsv|store in output_dir||
|Porcess in to a file|store in renewed_data_dir|use `pd.concat()` `drop_duplicates`|
|Save session|---|json|

In [2]:
import pandas as pd
from bs4 import BeautifulSoup as bs
import requests
from datetime import date, timedelta
from pathlib import Path
import io, zipfile
import json
from concurrent.futures import ThreadPoolExecutor


In [6]:
url = 'https://x.com/i/communitynotes/download-data'
r = requests.get(url)
print(r.status_code)
soup = bs(r.content)

200


In [ ]:
soup.find_all()

<!DOCTYPE html>
<html data-app-env="prod" data-app-version="cfeecfb0aaf81c1fa9609be665bc5f7ec2d55fbc" dir="ltr" lang="en"><head><meta charset="utf-8" nonce="R/v9n/+z3Gq96WnaQ40FeQ=="/><meta content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=0,viewport-fit=cover" name="viewport" nonce="R/v9n/+z3Gq96WnaQ40FeQ=="/><script async="" nonce="R/v9n/+z3Gq96WnaQ40FeQ==" src="https://abs.twimg.com/x-web/x-web/entry-client-logged-out-B2FW-W8t.js" type="module"></script><meta content="light dark" name="color-scheme"/><meta content="#FFFFFF" media="(prefers-color-scheme: light)" name="theme-color"/><meta content="#000000" media="(prefers-color-scheme: dark)" name="theme-color"/><meta content="R/v9n/+z3Gq96WnaQ40FeQ==" property="csp-nonce"/><title>X - The Everything App / X</title><meta content="2231777543" nonce="R/v9n/+z3Gq96WnaQ40FeQ==" property="fb:app_id"/><meta content="X (formerly Twitter)" nonce="R/v9n/+z3Gq96WnaQ40FeQ==" property="og:site_name"/><meta content="en_US" n

In [4]:
dataname_dict = {
    'notes': 'notes',
    'noteRatings': 'ratings',
    'noteStatusHistory': 'noteStatusHistory',
    'userEnrollment': "userEnrollment",
    'batSignals': 'batSignals'
            }

def get_notes_data(day: date, dataset_name: str, dataset_subname: str, out_dir="community_notes", max_iter = 100):
    extracted = []
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for i in range(max_iter):
        url = (f"https://ton.twimg.com/birdwatch-public-data/{day:%Y/%m/%d}/{dataset_name}/{dataset_subname}-0000{i}.zip")
        r = requests.get(url, timeout=60)
        if r.status_code != 200:
            print(f'num{i}, {r.status_code}')
            break
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall(out_dir)
            extracted.extend(z.namelist())
    return extracted



In [28]:
d = date.today()
for k, v in dataname_dict.items():
    for _ in range(14):
        if (hit := get_notes_data(d,k,v)):
            print("Downloaded:", hit)
            break
        d -= timedelta(days=1)

num5, 404
Downloaded: ['notes-00000.tsv', 'notes-00001.tsv', 'notes-00002.tsv', 'notes-00003.tsv', 'notes-00004.tsv']
num9, 404
Downloaded: ['ratings-00000.tsv', 'ratings-00001.tsv', 'ratings-00002.tsv', 'ratings-00003.tsv', 'ratings-00004.tsv', 'ratings-00005.tsv', 'ratings-00006.tsv', 'ratings-00007.tsv', 'ratings-00008.tsv']
num1, 404
Downloaded: ['noteStatusHistory-00000.tsv']
num1, 404
Downloaded: ['userEnrollment-00000.tsv']
num1, 404
Downloaded: ['batSignals-00000.tsv']


In [ ]:
def find_and_download(k, v):
    d = date.today()   # ← 每個資料集各自從今天開始找
    hit = get_notes_data(d, k, v)
    if hit:
        print("Downloaded:", hit)
    else:
        print(f'Today {d}, No data for {v}')
    return hit

with ThreadPoolExecutor(max_workers=5) as ex:
    results = dict(zip(dataname_dict, ex.map(lambda kv: find_and_download(*kv), dataname_dict.items())))

In [ ]:
def concat_and_process(name:str, day: date = date.today(), notes_path=Path('community_notes'), out_dir="Concat", max_iter = 100):  # I need speed so I use pyarrow
    data_list = list()
    for i in range(max_iter):
        print(f'start finding data {i} ...')
        try:
            df = pd.read_csv(f'{notes_path}/{name}-0000{i}.tsv', sep='\t', engine='pyarrow')
            data_list.append(df)
            print(f'Found {name}-0000{i}.tsv')
        except FileNotFoundError as e:
            print(e)
            break

    print('-'*20)
    print('File has been found, start concating...')
    try:
        concat_dataset = pd.concat(data_list, axis=0, ignore_index=True)
    except Exception as e:
        print('Concat process error:', e)
        return None

    print('-'*20)
    print('finish concating! Droping duplicates ...')
    try:
        dropdu_data = concat_dataset.drop_duplicates(keep="first")
        print(f"New data:{len(dropdu_data):,}")
    except Exception as e:
        print('Drop duplicates error: ', e)
        return None
    
    print('-'*20)
    print(f'Start dropping past duplicates ...')
    def drop_inpast(new_df = dropdu_data, name=name, data_dir=out_dir): # I only leave new dataset and store them as separated file with date
        yesterday = day - timedelta(days=1)
        # old_df = pd.read_parquet(f'{data_dir}/{name}_{yesterday:%Y-%m-%d}.parquet', engine='pyarrow')
        old_path = f'{data_dir}/{name}_{yesterday:%Y-%m-%d}.parquet'  # ←

        if name == 'ratings':  # ← ratings 太大,改用時間切點,只存新增的部分
            cutoff = pd.read_parquet(old_path, columns=['createdAtMillis'], engine='pyarrow')['createdAtMillis'].max()
            new_df = new_df[new_df['createdAtMillis'] > cutoff]
            print(f"Cutoff: {pd.to_datetime(cutoff, unit='ms')}, remained new data:{len(new_df):,}")
            return new_df

        old_df = pd.read_parquet(old_path, engine='pyarrow')  # ←
        print(f"Old data:{len(old_df):,}, yesterday= {yesterday}")

        old_hash = pd.util.hash_pandas_object(old_df, index=False)
        new_hash = pd.util.hash_pandas_object(new_df, index=False)
        new_df = new_df[(~new_hash.isin(old_hash) & ~new_hash.duplicated()).values]
        print(f"Remained new data:{len(new_df):,}")
        return pd.concat([old_df, new_df], ignore_index=True)

    try:
        new_data = drop_inpast()
    except Exception as e:
        print('Drop past duplicates FAILED: ',e)
        new_data = dropdu_data

    print('-'*20)
    print(f'Start writing file to {name}_{day:%Y-%m-%d}.parquet ...')
    Path(out_dir).mkdir(parents=True, exist_ok=True)  
    out_path = Path(out_dir) / f'{name}_{day:%Y-%m-%d}.parquet' # lower disk space
    new_data.to_parquet(out_path, index=False)

    return out_path  


In [7]:
for v in dataname_dict.values():
    out_path = concat_and_process(v)
    print(out_path)

start finding data 0 ...
Found notes-00000.tsv
start finding data 1 ...
Found notes-00001.tsv
start finding data 2 ...
Found notes-00002.tsv
start finding data 3 ...
Found notes-00003.tsv
start finding data 4 ...
Found notes-00004.tsv
start finding data 5 ...
[Errno 2] No such file or directory: 'community_notes/notes-00005.tsv'
--------------------
File has been found, start concating...
--------------------
finish concating! Droping duplicates ...
New data:3,157,743
--------------------
Start dropping past duplicates ...
Drop past duplicates FAILED:  [Errno 2] No such file or directory: 'Concat/notes_2026-10-07.parquet'
--------------------
Start writing file to notes_2026-10-08.parquet ...
Concat/notes_2026-10-08.parquet
start finding data 0 ...
Found ratings-00000.tsv
start finding data 1 ...
Found ratings-00001.tsv
start finding data 2 ...
Found ratings-00002.tsv
start finding data 3 ...
Found ratings-00003.tsv
start finding data 4 ...
Found ratings-00004.tsv
start finding data 5 

In [ ]:
import os
from pathlib import Path

print(os.getcwd())                                  
print(list(Path('community_notes').glob('*.tsv')))   

/Users/estebanbigmac/Desktop/My huge Python/RA Sameer
[PosixPath('community_notes/batSignals-00000.tsv'), PosixPath('community_notes/ratings-00003.tsv'), PosixPath('community_notes/noteStatusHistory-00000.tsv'), PosixPath('community_notes/ratings-00002.tsv'), PosixPath('community_notes/ratings-00000.tsv'), PosixPath('community_notes/ratings-00001.tsv'), PosixPath('community_notes/ratings-00005.tsv'), PosixPath('community_notes/ratings-00004.tsv'), PosixPath('community_notes/ratings-00006.tsv'), PosixPath('community_notes/ratings-00007.tsv'), PosixPath('community_notes/notes-00004.tsv'), PosixPath('community_notes/ratings-00008.tsv'), PosixPath('community_notes/notes-00002.tsv'), PosixPath('community_notes/notes-00003.tsv'), PosixPath('community_notes/notes-00001.tsv'), PosixPath('community_notes/notes-00000.tsv'), PosixPath('community_notes/userEnrollment-00000.tsv')]


In [ ]:
def clean_new_data(name:str, notes_path=Path('Concat'), out_dir="Process"):
    last_data = pd.read_parquet(f'{notes_path}/{name}.parquet', engine='pyarrow')
    

In [9]:
print(sorted(Path('Concat').rglob('*')))

[PosixPath('Concat/.DS_Store'), PosixPath('Concat/batSignals.parquet'), PosixPath('Concat/batSignals_2026-10-07.parquet'), PosixPath('Concat/noteStatusHistory.parquet'), PosixPath('Concat/noteStatusHistory_2026-10-07.parquet'), PosixPath('Concat/notes.parquet'), PosixPath('Concat/notes_2026-10-07.parquet'), PosixPath('Concat/ratings.parquet'), PosixPath('Concat/ratings_2026-10-07.parquet'), PosixPath('Concat/userEnrollment.parquet'), PosixPath('Concat/userEnrollment_2026-10-07')]


In [ ]:
Path('Concat/userEnrollment_2026-10-07').rename('Concat/userEnrollment_2026-10-07.parquet')

PosixPath('Concat/userEnrollment_2026-10-07.parquet')

: 